In [2]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import io

# 文件路径
gt_masks_path = '/home/pxl/myProject/血管分割/molong-深度插值/code/gt_masks.npy'
scores_path = '/home/pxl/myProject/血管分割/molong-深度插值/code/scores.npy'

# 读取保存的 numpy 数组文件
gt_masks = np.load(gt_masks_path)  # 加载真实标签数据
scores = np.load(scores_path)      # 加载预测结果数据

# 修改 gt_masks 和 scores 的形状，将最后一个维度转到第三个维度
gt_masks = np.transpose(gt_masks, (1, 2, 0))
scores = np.transpose(scores, (1, 2, 0))
print("gt_masks 和 scores 文件已成功加载。")
print(f"gt_masks 的形状: {gt_masks.shape}")
print(f"scores 的形状: {scores.shape}")

cal_idx = np.arange(0, 15)
print(cal_idx)
val_idx = np.setdiff1d(np.arange(1, scores.shape[2]), cal_idx) # 最后剩下的用于校准集
print(val_idx)

# 提取校准集的 score 数据
cal_scores = scores[:, :, cal_idx]  # 索引从 0 开始，cal_idx 减去 1
val_scores = scores[:, :, val_idx]  # 索引从 0 开始，cal_idx 减去 1
cal_gt_masks = gt_masks[:, :, cal_idx]  # 索引从 0 开始，cal_idx 减去 1
val_gt_masks = gt_masks[:, :, val_idx]  # 索引从 0 开始，cal_idx 减去 1


def create_segmentation_gif(image, initial_threshold=0.9, final_threshold=0.1, steps=20, save_path='segmentation_process.gif'):
    # 生成一系列阈值，逐步降低
    thresholds = np.linspace(initial_threshold, final_threshold, steps)
    
    # 用于存储所有帧的列表
    frames = []

    # 生成每个阈值的分割图像并保存到 frames 中
    for threshold in thresholds:
        # 生成当前阈值下的分割结果
        mask = image > threshold

        # 创建带透明度的颜色分割叠加图
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(image, cmap='gray', alpha=0.6)  # 原始图像
        ax.imshow(mask, cmap='jet', alpha=0.4)  # 分割结果叠加

        # 设置标题和去除坐标轴
        ax.set_title(f'Threshold: {threshold:.2f}')
        ax.axis('off')
        
        # 将当前图保存为图像格式
        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        buf.seek(0)
        frame = Image.open(buf)
        frames.append(frame)
        plt.close(fig)
    
    # 保存所有帧为 GIF
    frames[0].save(save_path, format='GIF', append_images=frames[1:], save_all=True, duration=200, loop=0)
    print(f'GIF 已保存至 {save_path}')

# 示例调用
# 假设 val_scores[:, :, no] 是分割的原始概率图像
no = 2
create_segmentation_gif(val_scores[:, :, no], initial_threshold=0.1, final_threshold=0, steps=10, save_path='./segmentation_process2.gif')

gt_masks 和 scores 文件已成功加载。
gt_masks 的形状: (584, 565, 20)
scores 的形状: (584, 565, 20)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
[15 16 17 18 19]
GIF 已保存至 ./segmentation_process2.gif
